In [1]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [2]:
vocab_size = 28

d_model = 32
context_length = 40
num_heads = 4
ffn_dim = 128

print("TensorFlow version:", tf.__version__)
print("Vocabulary size:", vocab_size)
print("Embedding dimension:", d_model)
print("Number of attention heads:", num_heads)
print("Dimension per head:", d_model // num_heads)

TensorFlow version: 2.21.0
Vocabulary size: 28
Embedding dimension: 32
Number of attention heads: 4
Dimension per head: 8


In [3]:
token_embedding = layers.Embedding(
    input_dim=vocab_size,
    output_dim=d_model
)

position_embedding = layers.Embedding(
    input_dim=context_length,
    output_dim=d_model
)

# Example batch: 2 sequences, each containing 40 token IDs
dummy_tokens = tf.random.uniform(
    shape=(2, context_length),
    minval=0,
    maxval=vocab_size,
    dtype=tf.int32
)

positions = tf.range(context_length)

token_vectors = token_embedding(dummy_tokens)
position_vectors = position_embedding(positions)

x = token_vectors + position_vectors

print("Token IDs shape:", dummy_tokens.shape)
print("Token embeddings shape:", token_vectors.shape)
print("Position embeddings shape:", position_vectors.shape)
print("Combined embeddings shape:", x.shape)

Token IDs shape: (2, 40)
Token embeddings shape: (2, 40, 32)
Position embeddings shape: (40, 32)
Combined embeddings shape: (2, 40, 32)


In [4]:
attention_layer = layers.MultiHeadAttention(
    num_heads = num_heads,
    key_dim = d_model//num_heads
)

attention_output = attention_layer(
    query = x,
    key = x,
    value = x,
    use_causal_mask = True
)

print("Input Shape: ", x.shape)
print("Attention Output Shape: ",attention_output.shape)

Input Shape:  (2, 40, 32)
Attention Output Shape:  (2, 40, 32)


In [9]:
# Residual connection + LayerNorm after attention
residual_1 = layers.Add()([x, attention_output])
norm_1 = layers.LayerNormalization(epsilon=1e-6)(residual_1)

print("Residula Layer Shape: ", residual_1.shape)
print("Normalized Layer Shape: ",norm_1.shape)

Residula Layer Shape:  (2, 40, 32)
Normalized Layer Shape:  (2, 40, 32)


In [11]:

ffn = keras.Sequential([
    layers.Dense(ffn_dim, activation="relu"),
    layers.Dense(d_model)
])

ffn_output = ffn(norm_1)

print("FFN Input Shape : ",norm_1.shape)
print("FFN Output Shape : ",ffn_output.shape)

FFN Input Shape :  (2, 40, 32)
FFN Output Shape :  (2, 40, 32)


In [13]:
residual_2 = layers.Add()([norm_1, ffn_output])
block_output = layers.LayerNormalization(epsilon=1e-6)(residual_2)

print("Transformer block output:", block_output.shape)

Transformer block output: (2, 40, 32)


In [15]:
# Project each token representation to vocabulary scores
lm_head = layers.Dense(vocab_size)

logits = lm_head(block_output)

print("Transformer block output shape: ",block_output.shape)
print("Logits Shape: ", logits.shape)

Transformer block output shape:  (2, 40, 32)
Logits Shape:  (2, 40, 28)


In [19]:
class TransformerBlock(layers.Layer):
    def __init__(self, d_model, num_heads, ffn_dim, dropout=0.1):
        super().__init__()

        self.attention = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=d_model//num_heads
        )

        self.ffn = keras.Sequential([
            layers.Dense(ffn_dim, activation="relu"),
            layers.Dense(d_model)
        ])

        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)

        self.dropout1 = layers.Dropout(dropout)
        self.dropout2 = layers.Dropout(dropout)

    def call(self,x, training=False):
        attetion_output = self.attention(
            query=x,
            key=x,
            value=x,
            use_causal_mask=True,
            training=training
        )

        x = self.norm1(
            x + self.dropout1(attention_output, training=training)
        )

        ffn_output = self.ffn(x, training=training)

        x = self.norm2(
            x + self.dropout2(ffn_output, training=training)
        )

        return x

In [20]:
block =TransformerBlock(
    d_model=d_model,
    num_heads=num_heads,
    ffn_dim=ffn_dim
)

test_block = block(x, training=False)

print("Input Shape: ",x.shape)
print("Block Output Shape: ",test_block.shape)

Input Shape:  (2, 40, 32)
Block Output Shape:  (2, 40, 32)


In [25]:
class MiniGPT(keras.Model):
    def __init__(
        self,
        vocab_size,
        context_length,
        d_model,
        num_heads,
        ffn_dim,
        dropout=0.1
    ):
        super().__init__()

        self.context_length = context_length

        self.token_embedding = layers.Embedding(
            vocab_size, d_model
        )

        self.position_embedding = layers.Embedding(
            context_length, d_model
        )

        self.transformer_block = TransformerBlock(
            d_model=d_model,
            num_heads=num_heads,
            ffn_dim=ffn_dim,
            dropout=dropout
        )

        self.final_norm = layers.LayerNormalization(epsilon=1e-6)
        self.lm_head = layers.Dense(vocab_size)

    def call(self, token_ids, training=False):
        seq_len = tf.shape(token_ids)[1]

        position = tf.range(seq_len)

        x = self.token_embedding(token_ids)
        x = x + self.position_embedding(position)

        x = self.transformer_block(x, training=False)
        x = self.final_norm(x)

        logits = self.lm_head(x)

        return logits

In [26]:
model = MiniGPT(
    vocab_size=vocab_size,
    context_length=context_length,
    d_model=d_model,
    num_heads=num_heads,
    ffn_dim=ffn_dim
)

test_tokens = tf.random.uniform(
    shape = (2,context_length),
    minval = 0,
    maxval = vocab_size,
    dtype=tf.int32
)

test_logits = model(test_tokens, training=False)

print("Input shape:", test_tokens.shape)
print("Output shape:", test_logits.shape)
print("Total parameters:", model.count_params())

Input shape: (2, 40)
Output shape: (2, 40, 28)
Total parameters: 15868


In [27]:
model.summary()

Model: "mini_gpt_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding_6 (Embedding)              │ (2, 40, 32)                 │             896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ embedding_7 (Embedding)              │ (40, 32)                    │           1,280 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ transformer_block_4                  │ ?                           │          12,704 │
│ (TransformerBlock)                   │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ layer_normalization_15               │ (2, 40, 32)                 │              64 │
│ (LayerNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_18 (Dense)                     │ (2, 40, 28)                 │             924 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 15,868 (61.98 KB)

 Trainable params: 15,868 (61.98 KB)

 Non-trainable params: 0 (0.00 B)